# Research-Executor — index des notebooks de recherche

Ce projet est le véhicule d'import de la [QC Investment Strategy Library](https://www.quantconnect.com/learning/articles/investment-strategy-library) : huit stratégies distillées, chacune dans son notebook de recherche QuantBook, plus un exécuteur en série.

**Contenu du projet** :

| Élément | Rôle |
|---|---|
| `research_<strategie>.ipynb` (×8) | Une stratégie par notebook : univers, logique, backtest léger QuantBook |
| `runner.ipynb` | Exécution en série des 8 notebooks (driver `nbconvert`) |
| `main.py` | Materializer : embarque les 8 notebooks en JSON (dict `NOTEBOOKS`) pour déploiement QC Cloud |
| `research.ipynb` (ce notebook) | Index : introspection réelle du registre, exécuté dans l'environnement de recherche |

**Navigation** : [ReadMe du projet](README.md) · [runner](runner.ipynb) · stratégies listées ci-dessous.


## L'environnement d'exécution

L'index tourne dans le même environnement que les notebooks de recherche (image `quantconnect/research`, kernel Python du moteur Lean). La cellule suivante instancie un `QuantBook` — l'objet racine de l'API de recherche — pour attester que l'environnement est vivant, puis l'index parcourt le registre.


In [1]:
# Environnement de recherche Lean : QuantBook racine + runtime Python.
# Aucune donnee n'est telechargee ici (pas d'add_equity) : l'index lit le registre local.
import sys

qb = QuantBook()
print(f"Python {sys.version.split()[0]} | QuantBook instancie ({type(qb).__name__})")


Python 3.11.14 | QuantBook instancie (QuantBook)


## Registre des stratégies

Introspection réelle : chaque `research_*.ipynb` du projet est ouvert et analysé — titre, concept (ligne `**Concept**` de l'en-tête), taille, univers de tickers (appels `add_equity`/`AddEquity`). Le dossier du projet est localisé par marqueur (`main.py` + le premier notebook de stratégie), le répertoire de travail pouvant varier selon le montage du conteneur.


In [2]:
# Index du registre : analyse des notebooks research_*.ipynb du projet.
import glob, json, os, re

def find_project_dir():
    roots, p = [], os.getcwd()
    for _ in range(5):
        roots.append(p); p = os.path.dirname(p)
    roots += ["/Lean/Launcher/bin/Debug/Notebooks", "/Lean/Launcher/bin/Debug"]
    roots += glob.glob("/Lean/Launcher/bin/Debug/Projects/*")
    for r in roots:
        if os.path.exists(os.path.join(r, "main.py")) and            os.path.exists(os.path.join(r, "research_asset_class_momentum.ipynb")):
            return r
    raise FileNotFoundError(f"dossier projet introuvable depuis {os.getcwd()}")

PROJECT = find_project_dir()
ADD_EQ = re.compile(r"(?:add_equity|AddEquity)\s*\(\s*[\"']([A-Za-z0-9.\-/]+)[\"']")

TICKERS_LIST = re.compile(r'^\s*(?:tickers|universe|symbols|etfs|assets|pairs)\s*=\s*\[([^\]]*)\]', re.M)

def symbols_of(nb):
    src = "".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    syms = set(ADD_EQ.findall(src))
    for block in TICKERS_LIST.findall(src):
        syms.update(re.findall(r'"([^"]+)"', block))
    return syms

rows = []
for name in sorted(os.listdir(PROJECT)):
    if not (name.startswith("research_") and name.endswith(".ipynb")):
        continue
    nb = json.load(open(os.path.join(PROJECT, name), encoding="utf-8"))
    head = "".join(nb["cells"][0]["source"]) if nb["cells"] else ""
    title = next((l[2:].strip() for l in head.splitlines() if l.startswith("# ")), name)
    concept = next((l.split("**Concept**:", 1)[1].strip()
                    for l in head.splitlines() if "**Concept**" in l), "")
    src = next((l.split("](", 1)[1].rstrip(")") if "](" in l
                  else l.split("**Source**:", 1)[1].strip()
                  for l in head.splitlines() if "**Source**" in l), "")
    syms = sorted(symbols_of(nb))
    rows.append(dict(name=name, title=title, concept=concept, src=src,
                     ncells=len(nb["cells"]),
                     ncode=sum(1 for c in nb["cells"] if c["cell_type"] == "code"), syms=syms))

print(f"Projet : {os.path.basename(PROJECT)}  |  {len(rows)} notebooks de strategie")
print("=" * 100)
header = "notebook".ljust(38) + " " + "titre".ljust(32) + " cell  code  univers"
print(header)
print("-" * 100)
for r in rows:
    print(r["name"].ljust(38) + " " + r["title"][:32].ljust(32)
          + " " + str(r["ncells"]).rjust(4) + " " + str(r["ncode"]).rjust(5)
          + "  " + ",".join(r["syms"]))
print("=" * 100)
print("Total : " + str(sum(r["ncells"] for r in rows)) + " cellules dont "
      + str(sum(r["ncode"] for r in rows)) + " code ; "
      + str(len({s for r in rows for s in r["syms"]})) + " tickers distincts")


Projet : Notebooks  |  8 notebooks de strategie
notebook                               titre                            cell  code  univers
----------------------------------------------------------------------------------------------------
research_asset_class_momentum.ipynb    Asset Class Momentum - Research     8     5  BND,EFA,GSG,SPY,VNQ
research_commodity_term_structure.ipynb Commodity Term Structure - Resea    6     3  
research_defensive_etf_rotation.ipynb  Defensive ETF Rotation - Researc    8     5  BSV,QQQ,SPXL,SPY,SQQQ,TECL,TECS,TQQQ,UVXY
research_long_short_harvest.ipynb      Long-Short Volatility Harvest ML    7     4  GLD,SPY
research_macro_factor_rotation.ipynb   Macro Factor Rotation - Research    7     4  BND,GLD,SPY
research_piotroski_fscore.ipynb        Piotroski F-Score Quality Value     7     4  SPY
research_puppies_of_dow.ipynb          Puppies of the Dow - Research No    6     3  DIA,SPY
research_volatility_regime_ml.ipynb    Volatility Regime ML - Research    1

### Lecture du résultat

Le registre compte **8 notebooks de recherche**, 59 cellules dont 34 de code, et **17 tickers distincts** au total. Quelques lectures :

- **Taille** : entre 6 et 10 cellules par notebook (3 à 6 de code) — chaque stratégie tient en une session de recherche courte ; `research_volatility_regime_ml` est la plus fournie (10 cellules, 6 code).
- **Univers** : 7 notebooks sur 8 déclarent leurs tickers equity via `add_equity` (littéral ou liste `tickers = [...]` — l'extraction couvre les deux formes). `research_commodity_term_structure` reste vide : sa stratégie est portée par des **contrats futures**, pas par des equities.
- **Étalement** : `research_defensive_etf_rotation` pousse l'univers le plus large (9 tickers : equity, leveraged, inverse, volatilité, bonds) — c'est la logique « rotation conditionnelle » du concept, qui a besoin d'actifs de régimes opposés.
- **Base commune** : `SPY` apparaît dans 7 notebooks sur 8 — l'etf de référence sert d'ancre de marché pour la plupart des stratégies.

Note d'exécution : dans le conteneur de recherche, le projet est monté en `/Lean/Launcher/bin/Debug/Notebooks` — c'est le point de montage, pas le nom du dossier projet, d'où « Projet : Notebooks » en première ligne. Le localisateur par marqueur (`main.py` + premier notebook de stratégie) retient ce répertoire quel que soit le montage.


## Sources et mécanisme d'exécution

Chaque notebook de stratégie cite sa source (article QC Investment Strategy Library) dans son en-tête. Pour exécuter l'ensemble en série, voir [`runner.ipynb`](runner.ipynb) ; pour déployer sur QC Cloud, `main.py` materialise les notebooks embarqués.

## Exercice — étendre l'index

**Énoncé** : ajoutez au tableau du registre une colonne « exercices » comptant les cellules stub (`# TODO` ou `Exercice a completer`) de chaque notebook de stratégie.

**Indices** :
- Reprenez `code_reg` et étendez le dict `rows` d'un champ `nex`
- Un stub se détecte par `'# TODO' in src or 'Exercice a completer' in src` sur la source d'une cellule code


In [3]:
# Exercice a completer : colonne exercices du registre
# Etape 1 : reprendre la boucle de la cellule du registre
# Etape 2 : compter les cellules code stub (# TODO / Exercice a completer)
# Etape 3 : afficher le tableau etendu
